# Data preprocess

In [ ]:
from pathlib import Path
import numpy as np
import snapatac2 as snap
from gcell._settings import get_setting
import scanpy as sc
import pandas as pd
from pyranges import PyRanges as pr
from preprocess_utils import (
    add_atpm,
    add_exp,
    create_peak_motif,
    download_motif,
    get_motif,
    query_motif,
)
import logging

# Configuration log
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),  
        logging.FileHandler('process.log')  
    ]
)
logger = logging.getLogger(__name__)

# Get annotation_dir 
annotation_dir = get_setting('annotation_dir')
logger.info(f"gcell currently using annotation directory: {annotation_dir}")

# # read ATAC data
# logger.info("Loading ATAC data from ../seacells/process/atac.h5ad")
# ad_atac = sc.read_h5ad('../seacells/process/atac.h5ad')
# ad_atac.obs['cell_type'] = ad_atac.obs_names
# logger.info(f"ATAC data loaded with {ad_atac.n_obs} cells and {ad_atac.n_vars} peaks")

# # get peak name
# def get_peak_from_snapatac(atac: snap.AnnData):
#     logger.info("Extracting peak names from ATAC data")
#     peak_names = pd.DataFrame(atac.var.index.str.split('-').tolist(), columns=['Chromosome', 'Start', 'End'])
#     peak_names['Start'] = peak_names['Start'].astype(int)
#     peak_names['End'] = peak_names['End'].astype(int)
#     logger.info(f"Extracted {peak_names.shape[0]} peaks")
#     return peak_names

# peaks = get_peak_from_snapatac(ad_atac)
# logger.info(f"Peak dataframe shape: {peaks.shape}")

# # Get a specific cell type peak ACPM
# def get_peak_acpm_for_cell_type(atac: snap.AnnData, cell_type: str):
#     logger.info(f"Processing peaks for cell type: {cell_type}")
#     peaks = get_peak_from_snapatac(atac)
#     counts = np.array(atac[atac.obs.cell_type == cell_type].X.sum(0)).flatten()
#     acpm = np.log10(counts / counts.sum() * 1e5 + 1)
#     peaks['aCPM'] = acpm / acpm.max()
#     peaks = peaks.query('Chromosome.str.startswith("chr") & ~Chromosome.str.endswith("M") & ~Chromosome.str.endswith("Y") & ~Chromosome.str.startswith("chrUn")')
#     peaks = pr(peaks, int64=True).sort().df
#     logger.info(f"Processed {peaks.shape[0]} peaks for {cell_type}")
#     return peaks

# # Generate peak files for each cell type
# logger.info("Generating peak files for all cell types")
# for cell_type in ad_atac.obs['cell_type']:
#     peaks = get_peak_acpm_for_cell_type(ad_atac, cell_type)
#     output_file = f'./process_seacells/atac/{cell_type.replace(" ", "_").replace("/", "_").lower()}.atac.bed'
#     peaks.to_csv(output_file, sep='\t', index=False, header=False)
#     logger.info(f"Saved peaks to {output_file}")

# Read RNA data
logger.info("Loading RNA data from ../seacells/process/rna.h5ad")
rna = sc.read_h5ad('../seacells/process/rna.h5ad')
rna.obs['cell_type'] = rna.obs_names
logger.info(f"RNA data loaded with {rna.n_obs} cells and {rna.n_vars} genes")

# # Obtaining RNA TPMs for specific cell types
# def get_rna_for_cell_type(rna: snap.AnnData, cell_type: str):
#     logger.info(f"Processing RNA for cell type: {cell_type}")
#     counts = rna[rna.obs.cell_type == cell_type].X.sum(0)
#     counts = np.log10(counts / counts.sum() * 1e6 + 1)
#     counts = np.array(counts).flatten()
#     rna_tpm = pd.DataFrame(counts, columns=['TPM'])
#     rna_tpm['gene_name'] = rna.var.index
#     sorted_tpm = rna_tpm[['gene_name', 'TPM']].sort_values(by='gene_name', ascending=True)
#     logger.info(f"Processed RNA TPM for {cell_type}, shape: {sorted_tpm.shape}")
#     return sorted_tpm

# # Generate RNA files for each cell type
# logger.info("Generating RNA files for all cell types")
# for cell_type in rna.obs['cell_type']:
#     rna_tpm = get_rna_for_cell_type(rna, cell_type)
#     output_file = f'./process_seacells/rna/{cell_type.replace(" ", "_").replace("/", "_").lower()}.rna.csv'
#     rna_tpm.to_csv(output_file, index=False)
#     logger.info(f"Saved RNA TPM to {output_file}")

# built peak-motif data
logger.info("Creating peak-motif data")
peak_bed = "./process_seacells/atac/10x_datasets_1-1@108.atac.bed"
get_motif_output = './process_seacells/get_motif.bed'
create_peak_motif(get_motif_output, "./process_seacells/seacells.zarr", peak_bed)
logger.info("Peak-motif data created in ./process_seacells/seacells.zarr")

# daa ATPM data
logger.info("Adding ATPM data for all cell types")
for cell_type in rna.obs['cell_type']:
    cell_type_key = cell_type.replace(" ", "_").replace("/", "_").lower()
    atac_file = f"./process_seacells/atac/{cell_type_key}.atac.bed"
    add_atpm("./process_seacells/seacells.zarr", atac_file, cell_type_key)
    logger.info(f"Added ATPM for {cell_type_key} from {atac_file}")

# add expression data
logger.info("Adding expression data for all cell types")
for cell_type in rna.obs['cell_type']:
    cell_type_key = cell_type.replace(" ", "_").replace("/", "_").lower()
    rna_file = f"./process_seacells/rna/{cell_type_key}.rna.csv"
    atac_file = f"./process_seacells/atac/{cell_type_key}.atac.bed"
    add_exp(
        "./process_seacells/seacells.zarr",
        rna_file,
        atac_file,
        cell_type_key,
        assembly="hg38",
        version=44,
        extend_bp=300,
        id_or_name="gene_name"
    )
    logger.info(f"Added expression data for {cell_type_key} from {rna_file} and {atac_file}")

# Fine tuning

In [ ]:
from pathlib import Path
import numpy as np
import os
import logging
from get_model.config.config import load_config_from_yaml
from get_model.run_region import run_zarr as run
from pytorch_lightning import Trainer
import torch

# os.environ["WANDB_API_KEY"] = 'YOUR_API_KEY'
# os.environ["WANDB_MODE"] = "offline"

# Configure logging for debugging and monitoring
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Load the list of cells
cell = os.listdir('./process_seacells/seacells.zarr/atpm/')
del cell[1]
n_total = len(cell)
n_per_chunk = n_total // 20  # Base size for each chunk
remainder = n_total % 20     # Remainder cells assigned to the first few chunks

# Load configuration and set a unified output directory
cfg = load_config_from_yaml("./training_config.yaml")
cfg.run.project_name = 'all_training'  # Project name
cfg.training.warmup_epochs = 0         # Avoid learning rate scheduler issues
logger.info(f"Starting training, planned for 10 epochs, with 20 chunks per epoch")

# Train for 10 epochs
# Note: range(2, 3) implies running only the 3rd epoch (index 2). Adjust range(10) for full training.
for epoch in range(2, 3):
    # Set random seed for each epoch
    np.random.seed(epoch)
    torch.manual_seed(epoch)
    cfg.run.run_name = f'epoch_{epoch+1}_full'  # Set a unique run name for each epoch
    
    # Randomly shuffle the cell list for each epoch
    shuffled_cell = np.random.permutation(cell)
    logger.info(f"Epoch {epoch+1}/10 - Shuffling cell order using seed {epoch}")
    
    # Split into 20 chunks based on the shuffled cell list
    chunks = []
    start = 0
    for i in range(20):
        chunk_size = n_per_chunk + (1 if i < remainder else 0)
        end = start + chunk_size
        chunks.append(shuffled_cell[start:end].tolist())
        start = end
    
    # Train 20 chunks and record cell names for each chunk
    # Note: range(18, 20) implies running only the last 2 chunks. Adjust range(20) for full training.
    for i in range(18, 20):
        logger.info(f"Epoch {epoch+1}/10 - Training chunk {i+1}/20, containing {len(chunks[i])} cells: {', '.join(chunks[i])}")
        cfg.dataset.celltypes = ','.join(chunks[i])
        
        # Determine which checkpoint to load
        if epoch == 0 and i == 0:
            # First chunk of the first epoch: Load default pre-trained checkpoint
            logger.info(f"Loading default pre-trained checkpoint: {cfg.finetune.checkpoint}")
        elif i == 0:
            # First chunk of subsequent epochs: Load final.ckpt from the previous epoch
            prev_final_checkpoint = f"{cfg.machine.output_dir}/{cfg.run.project_name}/epoch_{epoch}_full/checkpoints/final.ckpt"
            cfg.finetune.checkpoint = prev_final_checkpoint
            logger.info(f"Loading final checkpoint from previous epoch: {prev_final_checkpoint}")
        else:
            # Subsequent chunks within the same epoch: Load the latest versioned last.ckpt
            checkpoint_dir = f"{cfg.machine.output_dir}/{cfg.run.project_name}/{cfg.run.run_name}/checkpoints"
            checkpoint_files = list(Path(checkpoint_dir).glob("last*.ckpt"))
            if checkpoint_files:
                latest_checkpoint = max(checkpoint_files, key=os.path.getctime)
                cfg.finetune.checkpoint = str(latest_checkpoint)
                logger.info(f"Loading latest versioned checkpoint: {cfg.finetune.checkpoint}")
            else:
                logger.warning("No versioned checkpoint found, continuing training with last known state")
        
        # Train the current chunk
        trainer = run(cfg)

    # Save the final checkpoint for the current epoch
    final_checkpoint = f"{cfg.machine.output_dir}/{cfg.run.project_name}/{cfg.run.run_name}/checkpoints/final.ckpt"
    Path(final_checkpoint).parent.mkdir(parents=True, exist_ok=True)  # Ensure directory exists
    trainer.save_checkpoint(final_checkpoint)
    logger.info(f"Epoch {epoch+1}/10 completed, final checkpoint saved to {final_checkpoint}")
    
logger.info("Training for all 10 epochs completed")

# Performance evaluation

In [1]:
import scanpy as sc
import random
import pandas as pd

## Data input

In [18]:
adata = sc.read_h5ad('/pool1/qiaolibo/R_Project/model/seacells/process/rna.h5ad')
adata

AnnData object with n_obs × n_vars = 21795 × 36601
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness'

In [42]:
len(pd.DataFrame(adata.obs['celltype'].value_counts()))

36

In [11]:
# Define the number of samples to be sampled for each cell type
n_samples = 3

# Use groupby().apply() to operate on each group
# lambda x: ... defines an anonymous function, where x represents the sub-DataFrame grouped by each cell type
# We check the number of rows in each group (len(x)), if it is greater than or equal to n_samples, randomly sample n_samples
# Otherwise (if the total number of cell types is less than n_samples), return the entire contents of the group
sampled_obs = adata.obs.groupby('celltype', group_keys=False).apply(
    lambda x: x.sample(n=n_samples) if len(x) >= n_samples else x
)

# Extract the row name (index) from the sampling result and convert it to a list
sampled_cell_list = sampled_obs.index.tolist()

/tmp/ipykernel_410847/3924606766.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sampled_obs = adata.obs.groupby('celltype', group_keys=False).apply(


In [20]:
sampled_cell_list

['Weng_Nature_2024_2-2@112',
 'Weng_Nature_2024_2-1@37',
 'Weng_Nature_2024_2-1@84',
 '10x_datasets_6-1@8',
 'Wimmers_Cell_2023-61_CC0216@82',
 'Wimmers_Cell_2023-32_CC0022@65',
 'Weng_Nature_2024_4-3@16',
 'Weng_Nature_2024_3-3@38',
 'Weng_Nature_2024_3-3@64',
 'Boukhaled_Nat_Immunol_2022-P3@22',
 'Cheong_Cell_2023_1-lgtd17@76',
 'Wimmers_Cell_2023-71_CC0594@22',
 'Wimmers_Cell_2023-47_CC0138@10',
 'Wimmers_Cell_2023-48_CC0138@100',
 'Cheong_Cell_2023_1-jcov72_5@81',
 '10x_datasets_2-1@64',
 'Cheong_Cell_2023_1-jcov111@30',
 'Wimmers_Cell_2023-60_CC0256@90',
 'Boukhaled_Nat_Immunol_2022-HD2@51',
 'Boukhaled_Nat_Immunol_2022-P3@133',
 'Wimmers_Cell_2023-60_CC0256@72',
 'Cheong_Cell_2023_1-jcov127@6',
 'Li_Nat_Biotechnol_2023-MV_2@55',
 'NeurIPS_2021-site1_donor2@73',
 'Marderstein_Nature_2024-TS21_16_A@30',
 'Weng_Nature_2024_4-3@52',
 'Cheong_Cell_2023_2-bmmcpie@39',
 'Li_Nat_Biotechnol_2023-MV_1@2',
 'Marderstein_Nature_2024-TS21_11_G@25',
 'Li_Nat_Biotechnol_2023-MV_2@74',
 'Marders

In [20]:
target_cell_types = ['HSC/MPP', 'ELP', 'CMP', 'MEMP']

hspc = adata[adata.obs['celltype'].isin(target_cell_types), :]
hspc

View of AnnData object with n_obs × n_vars = 1353 × 36601
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness'

In [43]:
hspc.obs_names

Index(['NeurIPS_2021-site3_donor6@14', 'NeurIPS_2021-site3_donor7@0',
       'Cheong_Cell_2023_1-jcov21@7', 'Cheong_Cell_2023_1-jcov21@10',
       'Cheong_Cell_2023_1-jcov21@11', 'Cheong_Cell_2023_1-jcov21@12',
       'Cheong_Cell_2023_1-jcov21@25', 'Cheong_Cell_2023_1-jcov21@27',
       'Cheong_Cell_2023_1-jcov52@22', 'Cheong_Cell_2023_1-jcov52@26',
       ...
       'Cheong_Cell_2023_1-jcov110@166', 'Cheong_Cell_2023_1-jcov110@168',
       'Cheong_Cell_2023_1-jcov110@170', 'Wei_Immuni_2023-YW4@129',
       'Wei_Immuni_2023-YW4@136', 'Wei_Immuni_2023-YW4@139',
       'Wei_Immuni_2023-YW4@152', 'Wei_Immuni_2023-YW4@169',
       'Wei_Immuni_2023-YW4@182', 'Wei_Immuni_2023-YW4@216'],
      dtype='object', length=1353)

In [21]:
# Define the number of samples to be sampled for each cell type
n_samples = 3

# Use groupby().apply() to operate on each group
# lambda x: ... defines an anonymous function, where x represents the sub-DataFrame grouped by each cell type
# We check the number of rows in each group (len(x)), if it is greater than or equal to n_samples, randomly sample n_samples
# Otherwise (if the total number of cell types is less than n_samples), return the entire contents of the group
sampled_obs = hspc.obs.groupby('celltype', group_keys=False).apply(
    lambda x: x.sample(n=n_samples) if len(x) >= n_samples else x
)

# Extract the row name (index) from the sampling result and convert it to a list
sampled_cell_list = sampled_obs.index.tolist()

/tmp/ipykernel_410847/3110457121.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sampled_obs = hspc.obs.groupby('celltype', group_keys=False).apply(


In [22]:
sampled_cell_list

['Weng_Nature_2024_4-2@39',
 'Weng_Nature_2024_5-3@126',
 'Weng_Nature_2024_4-3@33',
 'Cheong_Cell_2023_1-jcov101_2@45',
 'Li_Nat_Biotechnol_2023-MV_1@11',
 'Weng_Nature_2024_5-2@10',
 'Marderstein_Nature_2024-TS21_11_G@44',
 'Cheong_Cell_2023_1-HD_AD@41',
 'Weng_Nature_2024_4-2@5',
 'Li_Nat_Biotechnol_2023-MV_2@112',
 'Li_Nat_Biotechnol_2023-MV_2@118',
 'Li_Nat_Biotechnol_2023-MV_2@82']

## Data output

In [28]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from gcell.utils.causal_lib import get_subnet, plot_comm, preprocess_net

from get_model.config.config import export_config, load_config_from_yaml
from get_model.run_region import run_zarr as run

In [ ]:
from gcell.cell.celltype import GETHydraCellType

from get_model.config.config import load_config

### all

In [ ]:
adata = sc.read_h5ad('/pool1/qiaolibo/R_Project/model/seacells/process/rna.h5ad')

In [64]:
cfg = load_config_from_yaml("training_config-tf.yaml")
use_checkpoint = '/pool1/qiaolibo/R_Project/model/get/process_seacells/all_training/epoch_3_full/checkpoints/best-v20.ckpt'
cfg.stage = 'predict'
cfg.dataset.leave_out_chromosomes = None
cfg.finetune.resume_ckpt = use_checkpoint
cfg.run.use_wandb=False
cfg.task.layer_names = []
cfg.task.gene_list = None
cfg.run.run_name='all'
cfg.dataset.leave_out_celltypes = celltype
cfg.dataset.zarr_path = '/pool1/qiaolibo/R_Project/model/get/process_seacells/seacells.zarr'
cfg.dataset.celltypes = ",".join([s.lower() for s in adata.obs_names.tolist()])

In [65]:
for celltype in [
 'weng_nature_2024_2-2@112',
 'weng_nature_2024_2-1@37',
 'weng_nature_2024_2-1@84',
 '10x_datasets_6-1@8',
 'wimmers_cell_2023-61_cc0216@82',
 'wimmers_cell_2023-32_cc0022@65',
 'weng_nature_2024_4-3@16',
 'weng_nature_2024_3-3@38',
 'weng_nature_2024_3-3@64',
 'boukhaled_nat_immunol_2022-p3@22',
 'cheong_cell_2023_1-lgtd17@76',
 'wimmers_cell_2023-71_cc0594@22',
 'wimmers_cell_2023-47_cc0138@10',
 'wimmers_cell_2023-48_cc0138@100',
 'cheong_cell_2023_1-jcov72_5@81',
 '10x_datasets_2-1@64',
 'cheong_cell_2023_1-jcov111@30',
 'wimmers_cell_2023-60_cc0256@90',
 'boukhaled_nat_immunol_2022-hd2@51',
 'boukhaled_nat_immunol_2022-p3@133',
 'wimmers_cell_2023-60_cc0256@72',
 'cheong_cell_2023_1-jcov127@6',
 'li_nat_biotechnol_2023-mv_2@55',
 'neurips_2021-site1_donor2@73',
 'marderstein_nature_2024-ts21_16_a@30',
 'weng_nature_2024_4-3@52',
 'cheong_cell_2023_2-bmmcpie@39',
 'li_nat_biotechnol_2023-mv_1@2',
 'marderstein_nature_2024-ts21_11_g@25',
 'li_nat_biotechnol_2023-mv_2@74',
 'marderstein_nature_2024-ts21_11_g@76',
 'marderstein_nature_2024-ts21_11_g@16',
 'marderstein_nature_2024-ts21_16_a@67',
 'marderstein_nature_2024-ts21_11_g@0',
 'thomson_nat_immunol_2024-b076_p1_pb00599_02@271',
 'oguchi_science_2024-d2_cd4bulk_2@48',
 'wimmers_cell_2023-73_cc0710@95',
 'li_nat_biotechnol_2023-mv_1@45',
 'li_nat_biotechnol_2023-mv_2@56',
 'cheong_cell_2023_1-jcov72_5@6',
 'wimmers_cell_2023-51_cc0191@46',
 'weng_nature_2024_6-bmmc@134',
 'wimmers_cell_2023-64_cc0294@111',
 'mimitou_nat_biotechnol_2021-dig_stim@43',
 'mimitou_nat_biotechnol_2021-dig_stim@117',
 'mimitou_nat_biotechnol_2021-lll_ctrl@48',
 'weng_nature_2024_6-bmmc@26',
 'neurips_2021-site4_donor8@6',
 'weng_nature_2024_4-2@109',
 'li_nat_biotechnol_2023-mv_2@112',
 'li_nat_biotechnol_2023-mv_2@21',
 'li_nat_biotechnol_2023-mv_2@17',
 'li_nat_biotechnol_2023-mv_2@79',
 'li_nat_biotechnol_2023-mv_2@90',
 'cheong_cell_2023_1-jcov69_8@48',
 'neurips_2021-site1_donor1@24',
 '10x_datasets_1-1@55',
 'boukhaled_nat_immunol_2022-p4@51',
 'weng_nature_2024_6-bmmc@114',
 'weng_nature_2024_6-bmmc@100',
 'neurips_2021-site1_donor2@41',
 'weng_nature_2024_1-2@15',
 'boukhaled_nat_immunol_2022-p2@30',
 'weng_nature_2024_1-2@46',
 'boukhaled_nat_immunol_2022-p8@58',
 'cheong_cell_2023_1-hdv@69',
 'boukhaled_nat_immunol_2022-p4@34',
 '10x_datasets_1-1@81',
 'cheong_cell_2023_1-jcov72_5@34',
 'wimmers_cell_2023-37_cc0120@78',
 'weng_nature_2024_6-bmmc@120',
 'weng_nature_2024_6-bmmc@141',
 'neurips_2021-site1_donor2@21',
 'wimmers_cell_2023-66_cc0350@49',
 'wimmers_cell_2023-31_cc0022@2',
 'thomson_nat_immunol_2024-b069_p1_pb00323_02@76',
 'thomson_nat_immunol_2024-b076_p1_pb00124_02@17',
 'weng_nature_2024_5-2@59',
 'weng_nature_2024_4-1@12',
 'weng_nature_2024_1-1@55',
 'thomson_nat_immunol_2024-b076_p1_pb00368_04@123',
 'wei_immuni_2023-yw8@80',
 'thomson_nat_immunol_2024-b076_p1_pb00599_02@317',
 'thomson_nat_immunol_2024-b076_p1_pb00124_02@68',
 'wimmers_cell_2023-39_cc0069@49',
 'thomson_nat_immunol_2024-b076_p1_imm19_445@21',
 'lareau_nat_genet_2023-dogma_pt1_rep2@45',
 'weng_nature_2024_2-2@31',
 'lareau_nat_genet_2023-dogma_pt1_rep1@120',
 'mimitou_nat_biotechnol_2021-dig_stim@41',
 'cheong_cell_2023_1-jcov72_5@84',
 'boukhaled_nat_immunol_2022-p4@57',
 '10x_datasets_4-1@27',
 'marderstein_nature_2024-ts21_16_b@28',
 'boukhaled_nat_immunol_2022-p6@22',
 'wimmers_cell_2023-35_cc0061@87',
 'thomson_nat_immunol_2024-b076_p1_pb00124_02@243',
 'thomson_nat_immunol_2024-b076_p1_pb00377_03@52',
 'cheong_cell_2023_1-jcov101@4',
 'wimmers_cell_2023-56_cc0249@72',
 'cheong_cell_2023_1-lgtd8m@76',
 'neurips_2021-site2_donor1@54'
]:
    zarr_path = './tf/tf/all/'+celltype+'.zarr'
    hydra_celltype = GETHydraCellType.from_config(cfg, 
                                              celltype=celltype, 
                                              zarr_path=zarr_path)
    hydra_celltype.gene_annot[hydra_celltype.gene_annot['Chromosome'] == 'chr11'].to_csv('./tf/evalu/all/'+celltype+'.csv')

### hspc

In [29]:
cfg = load_config_from_yaml("training_config-tf.yaml")
use_checkpoint = '/pool1/qiaolibo/R_Project/model/get/process_seacells/finetune_hspc/finetune_hspc_1353_chr11/checkpoints/best-v1.ckpt'
cfg.stage = 'predict'
cfg.dataset.leave_out_chromosomes = None
cfg.finetune.resume_ckpt = use_checkpoint
cfg.run.use_wandb=False
cfg.task.layer_names = []
cfg.task.gene_list = None
cfg.run.run_name='hspc'
cfg.dataset.leave_out_celltypes = celltype
cfg.dataset.zarr_path = '/pool1/qiaolibo/R_Project/model/get/process_seacells/seacells.zarr'
cfg.dataset.celltypes = ",".join([s.lower() for s in hspc.obs_names.tolist()])

In [41]:
for celltype in [
 'weng_nature_2024_4-2@39',
 'weng_nature_2024_5-3@126',
 'weng_nature_2024_4-3@33',
 'cheong_cell_2023_1-jcov101_2@45',
 'li_nat_biotechnol_2023-mv_1@11',
 'weng_nature_2024_5-2@10',
 'marderstein_nature_2024-ts21_11_g@44',
 'cheong_cell_2023_1-hd_ad@41',
 'weng_nature_2024_4-2@5',
 'li_nat_biotechnol_2023-mv_2@112',
 'li_nat_biotechnol_2023-mv_2@118',
 'li_nat_biotechnol_2023-mv_2@82'
]:
    zarr_path = './tf/tf/hspc/'+celltype+'.zarr'
    hydra_celltype = GETHydraCellType.from_config(cfg, 
                                              celltype=celltype, 
                                              zarr_path=zarr_path)
    hydra_celltype.gene_annot[hydra_celltype.gene_annot['Chromosome'] == 'chr11'].to_csv('./tf/evalu/hspc/'+celltype+'.csv')

## Data sorting

In [2]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import r2_score

# 1. Set the target directory
# Path('.') represents the current directory; modify to absolute or relative path as needed
base_dir = Path('./tf/evalu/all/')

# 2. Prepare a dictionary to store the final results
results_dict = {}

# 3. Check if the target directory exists
if not base_dir.is_dir():
    print(f"Error: Directory '{base_dir}' does not exist.")
else:
    # 4. Find and iterate through all .csv files in the directory
    csv_files = list(base_dir.glob('*.csv'))
    print(f"Found {len(csv_files)} CSV files in '{base_dir}', starting processing...")

    for file_path in csv_files:
        print(f"--- Processing: {file_path.name} ---")
        try:
            # Read the CSV file
            data = pd.read_csv(file_path)

            # Check if required columns exist
            required_columns = ['gene_name', 'pred', 'obs']
            if not all(col in data.columns for col in required_columns):
                print(f"  -> Warning: File {file_path.name} is missing required columns, skipped.")
                continue

            # Perform the specified core operations: Sort and Drop Duplicates
            processed_data = data.sort_values(
                by=['pred', 'obs'], 
                ascending=[False, False]
            ).drop_duplicates(
                subset=['gene_name'], 
                keep='first'
            )

            # Check if processed data is sufficient for calculation (at least 2 rows)
            if len(processed_data) < 2:
                print(f"  -> Warning: File {file_path.name} has fewer than 2 rows after processing, cannot calculate correlations, skipped.")
                continue

            # Calculate three metrics
            # a. Pearson Correlation
            pearson_corr = processed_data['obs'].corr(processed_data['pred'], method='pearson')
            
            # b. Spearman Rank Correlation
            spearman_corr = processed_data['obs'].corr(processed_data['pred'], method='spearman')
            
            # c. R-squared (Coefficient of Determination)
            r_squared = r2_score(processed_data['obs'], processed_data['pred'])
            
            # Store results in dictionary, using filename (without extension) as key
            file_name_stem = file_path.stem
            results_dict[file_name_stem] = [pearson_corr, spearman_corr, r_squared]
            
            print(f"  -> Processed successfully. Pearson: {pearson_corr:.4f}, Spearman: {spearman_corr:.4f}, R-squared: {r_squared:.4f}")

        except Exception as e:
            # Catch other possible errors, such as file reading failures
            print(f"  -> Error occurred while processing file {file_path.name}: {e}")

    # 5. Convert the results dictionary to the final DataFrame
    if results_dict:
        final_df = pd.DataFrame.from_dict(
            results_dict, 
            orient='index', 
            columns=['Pearson_Correlation', 'Spearman_Correlation', 'R_Squared']
        )
        
        print("\n\n--- All files processed ---")
        print("Final Results Summary:")
        print(final_df)
        
        # (Optional) Save results to a CSV file
        # final_df.to_csv('analysis_summary.csv')
        # print("\nResults saved to analysis_summary.csv")

    else:
        print("\nNo files were successfully processed.")

在 'tf/evalu/all' 中找到 102 个 CSV 文件，开始处理...
--- 正在处理: weng_nature_2024_2-2@112.csv ---
  -> 处理成功。Pearson: 0.8828, Spearman: 0.8718, R-squared: 0.7791
--- 正在处理: weng_nature_2024_2-1@37.csv ---
  -> 处理成功。Pearson: 0.8906, Spearman: 0.8857, R-squared: 0.7928
--- 正在处理: weng_nature_2024_2-1@84.csv ---
  -> 处理成功。Pearson: 0.8972, Spearman: 0.8886, R-squared: 0.8027
--- 正在处理: 10x_datasets_6-1@8.csv ---
  -> 处理成功。Pearson: 0.9021, Spearman: 0.8978, R-squared: 0.8124
--- 正在处理: wimmers_cell_2023-61_cc0216@82.csv ---
  -> 处理成功。Pearson: 0.9438, Spearman: 0.9346, R-squared: 0.8894
--- 正在处理: wimmers_cell_2023-32_cc0022@65.csv ---
  -> 处理成功。Pearson: 0.8785, Spearman: 0.8700, R-squared: 0.7710
--- 正在处理: weng_nature_2024_4-3@16.csv ---
  -> 处理成功。Pearson: 0.9067, Spearman: 0.8991, R-squared: 0.8188
--- 正在处理: weng_nature_2024_3-3@38.csv ---
  -> 处理成功。Pearson: 0.9431, Spearman: 0.9211, R-squared: 0.8886
--- 正在处理: weng_nature_2024_3-3@64.csv ---
  -> 处理成功。Pearson: 0.8819, Spearman: 0.8772, R-squared: 0.7611
---

In [3]:
final_df

,Pearson_Correlation,Spearman_Correlation,R_Squared
weng_nature_2024_2-2@112,0.882821,0.871790,0.779094
weng_nature_2024_2-1@37,0.890571,0.885700,0.792766
weng_nature_2024_2-1@84,0.897184,0.888640,0.802678
10x_datasets_6-1@8,0.902089,0.897809,0.812427
wimmers_cell_2023-61_cc0216@82,0.943772,0.934558,0.889434
...,...,...,...
thomson_nat_immunol_2024-b076_p1_pb00377_03@52,0.962583,0.948712,0.926176
cheong_cell_2023_1-jcov101@4,0.915849,0.905106,0.835914
wimmers_cell_2023-56_cc0249@72,0.901688,0.893545,0.812721
cheong_cell_2023_1-lgtd8m@76,0.857788,0.853943,0.724356


In [67]:
final_df.to_csv('./tf/evalu/all.csv')

## Visual

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# --- 1. Setup Parameters ---
# Path to the directory containing data
data_directory = Path('./tf/evalu/')
# Note: This script now automatically identifies all numeric columns in the CSVs, no need to specify manually
# DATA_COLUMNS = ['column_1', 'column_2', 'column_3'] 

# --- 2. Read and Consolidate all CSV Files ---
all_data_frames = []
csv_files = list(data_directory.glob('*.csv'))

if not csv_files:
    print(f"Error: No CSV files found in directory '{data_directory}'.")
else:
    print(f"Found {len(csv_files)} CSV files, starting processing...")
    for file_path in csv_files:
        try:
            # Read CSV file
            df = pd.read_csv(file_path, index_col=0)
            
            # Add a new column to identify the data source (filename)
            df['source_file'] = file_path.stem 
            all_data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")

# Concatenate all data into a single DataFrame
if all_data_frames:
    combined_df = pd.concat(all_data_frames, ignore_index=True)

    # --- 3. Reshape Data (Melt) ---
    # Convert wide-format data to long-format for plotting
    actual_data_columns = [col for col in combined_df.columns if col != 'source_file']
    
    long_form_df = combined_df.melt(
        id_vars=['source_file'],
        value_vars=actual_data_columns,
        var_name='data_column', # New column name for original column headers
        value_name='value'      # New column name for values
    )

    # --- 4. Plot Boxplot using Seaborn ---
    print("Data prepared, starting plotting...")
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(14, 8))

    # --- Main modification here: Swapped x and hue values ---
    # Now x-axis is source file, hue (color/grouping) is data column
    sns.boxplot(
        data=long_form_df,
        x='source_file',
        y='value',
        hue='data_column'
    )
    # ---------------------------------------------

    # --- 5. Customize Chart Aesthetics ---
    plt.title('Comparison of Data Columns Distribution Across File Sources', fontsize=16)
    plt.xlabel('File Source', fontsize=12)
    plt.ylabel('Value', fontsize=12)
    plt.xticks(rotation=15)
    plt.legend(title='Data Column') # Legend title is now "Data Column"
    plt.tight_layout()

    # (Optional) Save chart to file
    output_filename = 'boxplot_comparison.pdf'
    plt.savefig(output_filename, dpi=300)
    print(f"Chart saved as {output_filename}")
    
    # Show chart
    plt.show()

# Interpretation

## Compute

In [ ]:
import logging  
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from gcell.utils.causal_lib import get_subnet, plot_comm, preprocess_net

from get_model.config.config import export_config, load_config_from_yaml, load_config
from get_model.run_region import run_zarr as run

from gcell.cell.celltype import GETHydraCellType

import scanpy as sc
from gcell.cell.celltype import parallel_get_gene_by_motif

i = 0
for celltype in ['b_cells','b-cell_lineage','cycling_cells','dc','early_mk','erythroid','hsc_mpp','ilc','macrophages','mast_cells','monocytes','pdc','plasma_cells','t_cells']:
    i += 1
    try:
        cfg = load_config_from_yaml("training_config-tf.yaml")
        # use_checkpoint = f'{cfg.machine.output_dir}/{cfg.run.project_name}/{cfg.run.run_name}/checkpoints/best-v1.ckpt' 
        use_checkpoint = '/pool1/qiaolibo/R_Project/model/get/process_seacells/all_training/epoch_3_full/checkpoints/best-v20.ckpt'
        cfg.stage = 'predict'
        cfg.dataset.leave_out_chromosomes = None
        cfg.finetune.resume_ckpt = use_checkpoint
        cfg.run.use_wandb=False
        cfg.task.layer_names = []
        cfg.task.gene_list = None
        cfg.run.run_name='tf'
        cfg.dataset.leave_out_celltypes = celltype
        # trainer = run(cfg)
        
        zarr_path = './tf/tf/tf/'+celltype+'.zarr'
        hydra_celltype = GETHydraCellType.from_config(cfg, celltype=celltype, zarr_path=zarr_path)
        
        parallel_get_gene_by_motif(zarr_path, max_workers = 20)
        
        hydra_celltype.gene_by_motif.get_causal()
        
    except Exception as e:
        continue  

## Analysis

### motif search

In [125]:
hydra_celltype.motif.annotations[hydra_celltype.motif.annotations['Name'] == 'ZNF5']['Motif'].tolist()

[]

In [ ]:
## RUNX1
# EVI1/MECOM: MECOM
# ZNF140: ZNF140
# TBX/3: TBX3, TBX21, TBX2, EOMES
# HD/22: NKX3-1, NKX2-1, NKX2-5, NKX2-3, NKX2-8
# NR/19: NR1D1, NR1D2, RORA, RORB, RORC
# NFKB/1: NFKB1, NFKB2, REL, RELA, RELB

In [ ]:
## GFI1B
# 'CCAAT/CEBP': CEBPA, CEBPB, CEBPD, CEBPE, CEBPG, DBP, HLF, TEF, NFIL3
# 'NFAT/1': NFATC1, NFATC2, NFATC3, NFATC4
# 'NR/1': HNF4A, HNF4G, NR1H2, NR1H3, NR2C2, NR2F1, NR2F6, RXRA, RXRB, RXRG
# 'ZNF134': ZNF134
# 'ZNF418': ZNF418

In [ ]:
## GATA2
# 'HIC/1': HIC1, HIC2
# 'LEF1': LEF1, TCF7, ZNF350
# 'NR/11': ESRRB, NR5A2, NR6A1, RXRA, RARA, RXRG
# 'PRDM16': PRDM16
# 'ZNF143': IKZF1, ZNF76, ZNF143

In [ ]:
## EVI1/MECOM
# 'EBF1':  EBF1
# 'MEF2':  MEF2A, MEF2B, MEF2C, MEF2D
# 'NR/11':  ESRRB, NR5A2, NR6A1, RXRA, RARA, RXRG
# 'RBPJ':  RBPJ, ZNF75A
# 'SRF': SRF

In [ ]:
## ZNF521


In [126]:
hydra_celltype.motif.annotations[hydra_celltype.motif.annotations['Name'] == 'ZNF5']

,Cluster_ID,Motif,Database,Consensus,Relative_orientation,Width,Left_offset,Right_offset,Name,DBD,Seed_motif,Total_width,Consensus_left,Consensus_right,Cluster_size


In [133]:
hydra_celltype.motif.annotations.query('Motif.str.startswith("FOXG")')

,Cluster_ID,Motif,Database,Consensus,Relative_orientation,Width,Left_offset,Right_offset,Name,DBD,Seed_motif,Total_width,Consensus_left,Consensus_right,Cluster_size
1156,80,FOXG1_MA0613.1,Jaspar2018,GTAAACAA,+,8,5,9,FOX/5,forkhead,FOXC1_forkhead_1,22,5,17,51
1183,80,FOXG1_forkhead_1,Taipale_Cell_2013,ATAAACAATTGTAAACA,+,17,5,0,FOX/5,forkhead,FOXC1_forkhead_1,22,5,17,51
1882,203,FOXG1_forkhead_2,Taipale_Cell_2013,ACGGACACAATG,+,12,0,0,FOX/8,forkhead,FOXG1_forkhead_2,12,1,11,5


In [86]:
hydra_celltype.motif.annotations.query('Motif.str.startswith("TP53")')

,Cluster_ID,Motif,Database,Consensus,Relative_orientation,Width,Left_offset,Right_offset,Name,DBD,Seed_motif,Total_width,Consensus_left,Consensus_right,Cluster_size
2045,257,TP53_MA0106.3,Jaspar2018,GACATGCCCGGGCATGTT,-,18,1,1,P53-like/1,P53L,P53_HUMAN.H11MO.0.A,20,2,18,12


In [134]:
hydra_celltype.motif.annotations.to_csv('./process_seacells/motif_cluster.csv')

In [45]:
hydra_celltype.motif.annotations['Name'].value_counts().to_csv('motif_number.csv')

### interaction network

In [15]:
from gcell.utils.causal_lib import get_subnet, plot_comm, preprocess_net

In [109]:
g = hydra_celltype.gene_by_motif.get_causal()
g_ = preprocess_net(g, 0.24)

In [ ]:
plot_comm(get_subnet(g_.copy(), 'EVI1/MECOM'), figsize=(10, 10))

In [111]:
nodes = sorted(list(get_subnet(g_.copy(), 'EVI1/MECOM').nodes()), key=lambda x: str(x))
nodes

['EBF1', 'EVI1/MECOM', 'MEF2', 'NR/11', 'RBPJ', 'SRF']

In [ ]:
plot_comm(g_, figsize=(20, 20))

In [12]:
nodes = list(g_.nodes())
nodes

['AHR',
 'AP1/1',
 'AP1/2',
 'BCL6/2',
 'CCAAT/CEBP',
 'CREB/ATF/1',
 'CREB/ATF/2',
 'CTCF',
 'CUX/4',
 'DMRT1',
 'EBF1',
 'EGR',
 'ETS/1',
 'ETS/2',
 'EVI1/MECOM',
 'EWSR1/FLI1',
 'Ebox/CACCTG',
 'Ebox/CACGTG/1',
 'Ebox/CAGATGG',
 'Ebox/CATATG',
 'FEZF1',
 'FOX/3',
 'FOX/4',
 'FOX/5',
 'FOX/7',
 'FOX/8',
 'GATA',
 'GC-tract',
 'GCM',
 'GFI',
 'GLI',
 'GLIS',
 'GMEB2/2',
 'GMEB2/3',
 'GRHL',
 'HD/10',
 'HD/12',
 'HD/15',
 'HD/16',
 'HD/18',
 'HD/19',
 'HD/2',
 'HD/20',
 'HD/22',
 'HD/5',
 'HD/8',
 'HEN1',
 'HIC/1',
 'HIF',
 'HINFP1/1',
 'HINFP1/2',
 'HINFP1/3',
 'HLTF',
 'INSM1',
 'IRF/1',
 'IRF/2',
 'KLF/SP/1',
 'KLF/SP/2',
 'KLF/SP/3',
 'LEF1',
 'MEF2',
 'MFZ1',
 'MIES',
 'MTF1',
 'MYB/4',
 'MYB/5',
 'MZF1',
 'NFAC/2',
 'NFAT/1',
 'NFAT/2',
 'NFAT/4',
 'NFI/1',
 'NFI/2',
 'NFI/3',
 'NFKB/1',
 'NFKB/3',
 'NR/1',
 'NR/11',
 'NR/12',
 'NR/13',
 'NR/14',
 'NR/15',
 'NR/16',
 'NR/18',
 'NR/19',
 'NR/20',
 'NR/3',
 'NR/7',
 'NR/8',
 'OCT4+SOX2',
 'OVOL1',
 'P53-like/1',
 'PAX/1',
 'PAX/2',

In [19]:
import networkx as nx
import pickle

# Let's assume 'g' is your graph

# Save the graph object using pickle
with open("my_graph.pickle", "wb") as f:
    pickle.dump(g, f)

print("Graph successfully saved to my_graph.pickle")

Graph successfully saved to my_graph.pickle


### network visual

In [116]:
def get_color(i, r_off=1, g_off=1, b_off=1):
    """Assign a color to a vertex.

    Parameters
    ----------
    i: int
        The index of the vertex.
    r_off: int, optional
        The offset for the red channel, defaults to 1.
    g_off: int, optional
        The offset for the green channel, defaults to 1.
    b_off: int, optional
        The offset for the blue channel, defaults to 1.
    """
    n = 16
    low, high = 0.5, 0.9
    span = high - low
    r = low + span * (((i + r_off) * 3) % n) / (n - 1)
    g = low + span * (((i + g_off) * 5) % n) / (n - 1)
    b = low + span * (((i + b_off) * 7) % n) / (n - 1)
    return (r, g, b)


In [72]:
def plot_comm(G, figsize=(10, 10), title="Network structure", savefig=False, edge_scale_factor=1.5):
    """Plot the community structure of the graph.

    Parameters
    ----------
    G: nx.Graph
        The graph to plot.
    figsize: tuple, optional
        The size of the figure, defaults to (10, 10).
    title: str, optional
        The title of the plot, defaults to "Network structure".
    savefig: str, optional
        The path to save the plot, defaults to False.
    edge_scale_factor: float, optional
        The factor to scale edge widths, defaults to 1.5.
    """
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    # Set community color for internal edges
    external = [(v, w) for v, w in G.edges if G.edges[v, w]["community"] == 0]
    internal = [(v, w) for v, w in G.edges if G.edges[v, w]["community"] > 0]
    internal_color = [
        "#fddbc7" if (G.edges[e]["weight"] > 0) else "#d1e5f0" for e in internal
    ]
    external_color = [
        "#ef8a62" if (G.edges[e]["weight"] > 0) else "#67a9cf" for e in external
    ]
    node_color = [get_color(G.nodes[v]["community"]) for v in G.nodes]

    weights = [np.absolute(G[u][v]["weight"]) * edge_scale_factor for u, v in G.edges]  # 放大边粗细
    # external edges
    pos = nx.spring_layout(G)
    nx.draw(
        G,
        pos=pos,
        node_size=0,
        edgelist=external,
        width=weights,
        ax=ax,
        edge_color=external_color,
        node_color=node_color,
        with_labels=True,
    )
    # internal edges
    nx.draw(
        G,
        pos=pos,
        node_size=100,
        edgelist=internal,
        ax=ax,
        width=weights,
        edge_color=internal_color,
        node_color=node_color,
        with_labels=True,
    )
    ax.set_title(title)
    plt.tight_layout()
    # add padding in the margins
    plt.margins(0.3, 0.3)
    if savefig:
        save_path = Path(savefig) / f"{title}.pdf"
        plt.savefig(save_path, dpi=300)
    plt.show()
    plt.close()

In [73]:
import numpy as np
import networkx as nx

In [ ]:
plot_comm(g_, figsize=(20, 20), savefig='./',edge_scale_factor=1)

In [ ]:
plot_comm(get_subnet(g_.copy(), 'EVI1/MECOM'), figsize=(10, 10), savefig='./', edge_scale_factor=5)

In [ ]:
plot_comm(get_subnet(g_.copy(), 'RUNX/1'), figsize=(10, 10), savefig='./', edge_scale_factor=5)

In [ ]:
plot_comm(g_, figsize=(20, 20), savefig='./', edge_scale_factor=2)

### Gene annotation and correlation calculation

In [141]:
hydra_celltype.gene_annot

,gene_name,Chromosome,Start,End,Strand,pred,obs,accessibility
0,A1BG-AS1,chr19,58347400,58347904,0,1.048067,0.986405,1.0
1,A2M,chr12,9115590,9116090,1,0.010100,0.000000,1.0
2,A2ML1-AS1,chr12,8830744,8831244,1,0.001587,0.000000,1.0
3,A3GALT2,chr1,33321012,33321512,1,0.019750,0.000000,1.0
4,A4GALT,chr22,42720184,42720680,1,0.024706,0.157281,1.0
...,...,...,...,...,...,...,...,...
24922,ZYG11A,chr1,52842488,52842988,0,0.047970,0.000000,1.0
24923,ZYG11B,chr1,52726172,52726672,0,1.398046,1.248547,1.0
24924,ZYX,chr7,143380752,143381248,0,1.811213,1.756096,1.0
24925,ZZEF1,chr17,4142784,4143284,1,1.762756,1.736374,1.0


In [24]:
pearson_corr = hydra_celltype.gene_annot[['pred', 'obs']].corr(method='pearson')
print("Pearson:\n", pearson_corr)

Pearson相关系数矩阵:
           pred       obs
pred  1.000000  0.860027
obs   0.860027  1.000000


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(
    data=hydra_celltype.gene_annot,
    x='pred',
    color='blue',
    label='Predicted',
    kde=True,
    alpha=0.3
)

sns.histplot(
    data=hydra_celltype.gene_annot,
    x='obs',
    color='orange',
    label='Observed',
    kde=True,
    alpha=0.3
)

plt.title("Distribution Comparison")
plt.xlabel("Value")
plt.legend()
plt.show()

### motif-gene

In [50]:
import pandas as pd

In [51]:
?hydra_celltype.gene_jacobian_summary

Signature:
hydra_celltype.gene_jacobian_summary(
    gene,
    axis='motif',
    multiply_input=True,
    stats='absmean',
)
Docstring:
Summarizes the Jacobian for a given gene.

This function calculates the Jacobian for a given gene and summarizes it based on the specified axis.
If the axis is "motif", it concatenates the Jacobian summaries along axis 1 and then sums them.
If the axis is "region", it concatenates the Jacobian summaries along axis 0, groups them by index, chromosome, and start,
and then sums the scores.

Parameters
----------
gene
    The gene for which the Jacobian is to be calculated.
axis
    The axis along which to summarize the Jacobian. Defaults to "motif".
multiply_input
    If True, the input is multiplied. Defaults to True.

Returns
-------
pd.DataFrame: A DataFrame containing the summarized Jacobian.
File:      ~/miniconda3/envs/get/lib/python3.12/site-packages/gcell/cell/celltype.py
Type:      method

In [157]:
data = pd.DataFrame(hydra_celltype.gene_jacobian_summary('BCL11A', axis='motif')).sort_values(by=0, ascending=False)
data

,0
Accessibility,0.003533
ZFX,0.002593
CTCF,0.002558
HD/20,0.002264
TFAP2/1,0.001952
...,...
GMEB2/1,0.000013
HD/11,0.000013
MYB/2,0.000012
MYB/4,0.000010


#### motif-gene link

In [9]:
hydra_celltype.gene_by_motif.data

,AHR,AIRE,AP1/1,AP1/2,ARI5A,ARI5B,BATF,BCL6/1,BCL6/2,CCAAT/CEBP,...,ZNF667,ZNF680,ZNF708,ZNF713,ZNF768,ZNF784,ZNF85,ZSCAN3,ZSCAN4,Accessibility
A1BG-AS1,7.743214e-05,2.226985e-05,-1.383094e-04,-1.540129e-05,-1.301224e-05,7.044092e-06,5.135569e-05,-2.660735e-06,4.070098e-05,4.077330e-07,...,-1.016204e-04,0.000019,-8.897580e-05,6.847584e-06,-8.018938e-05,1.797162e-05,2.298947e-05,-1.259073e-04,3.438916e-05,-0.000284
A2M,-1.347388e-05,-3.813999e-06,5.975647e-05,-6.933920e-05,1.010573e-05,-6.394494e-06,-1.087168e-05,1.029927e-05,9.164574e-05,-9.876208e-05,...,8.072476e-05,0.000068,1.270156e-04,6.890961e-07,4.317053e-05,-2.126507e-05,3.827267e-05,8.531137e-05,-4.861793e-06,0.000134
A2ML1-AS1,-9.698515e-08,-1.514029e-07,-5.668064e-07,-3.577458e-07,1.193609e-07,-8.941944e-08,-1.683806e-07,1.043367e-07,-2.017498e-07,1.345688e-07,...,1.196772e-07,0.000002,1.163501e-07,-4.506572e-08,4.263912e-07,5.786667e-08,3.207646e-07,-4.354202e-07,2.453123e-07,-0.000005
A3GALT2,1.190235e-06,-3.942095e-06,2.269799e-06,6.438992e-06,8.620977e-07,-3.456246e-07,-5.531805e-07,-5.508346e-07,-1.216209e-05,-1.669771e-06,...,4.299893e-06,0.000009,4.029564e-06,8.201407e-07,2.011555e-05,1.726164e-06,1.972709e-06,-1.595637e-05,4.206376e-06,0.000091
A4GALT,-2.265079e-06,-1.737892e-06,1.046462e-05,-6.888577e-07,4.195994e-07,2.941680e-06,-7.929231e-06,1.184459e-06,-1.830517e-05,4.516942e-07,...,1.133716e-05,-0.000002,6.902020e-07,-2.649240e-06,1.808674e-05,-1.971981e-06,6.002678e-06,5.760076e-06,-9.799182e-06,0.000050
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYG11A,-1.377413e-06,-1.200087e-06,-1.235312e-05,-7.223572e-06,-1.112604e-07,-2.082164e-07,2.664478e-06,2.009739e-06,-2.162936e-06,-9.758697e-06,...,-1.204782e-06,-0.000004,-9.124031e-06,7.208389e-07,-1.004188e-05,1.516656e-06,5.647491e-06,-9.310552e-06,1.180697e-05,-0.000018
ZYG11B,-2.157167e-05,-2.786524e-05,4.967218e-05,-2.761724e-05,-1.085803e-04,-6.501054e-06,-2.329238e-06,-3.166604e-07,-5.355656e-06,-1.652355e-04,...,3.928353e-05,-0.000022,3.216930e-05,3.972638e-07,-1.371249e-05,-1.090281e-06,-4.135619e-06,5.567827e-05,-1.341400e-05,0.000002
ZYX,-1.504352e-05,3.585477e-06,7.031266e-05,-3.538172e-06,-3.523812e-05,-1.288512e-05,-4.713955e-05,7.316169e-06,2.307395e-05,-4.946159e-05,...,1.331376e-05,0.000029,6.122809e-06,2.339567e-05,1.287677e-05,-7.257033e-06,-1.040644e-05,-5.609778e-05,-3.750674e-05,0.000883
ZZEF1,1.234201e-05,-1.366671e-04,-8.966001e-05,1.776723e-05,-1.649890e-05,6.408886e-07,4.157687e-05,-5.516044e-07,1.324512e-05,4.402985e-05,...,-4.297127e-05,0.000108,-1.210865e-04,-1.204373e-05,-1.275159e-04,8.504911e-06,5.080851e-05,-1.424184e-04,-9.081258e-06,-0.000053


In [10]:
hydra_celltype.gene_by_motif.data.to_csv('./process_seacells/bulk_hsc_mpp_motif_gene.csv')

In [62]:
hydra_celltype.gene_by_motif.data.loc['MECOM',]

AHR              0.000023
AIRE            -0.000016
AP1/1           -0.000128
AP1/2           -0.000048
ARI5A            0.000003
                   ...   
ZNF784           0.000021
ZNF85            0.000060
ZSCAN3          -0.000110
ZSCAN4           0.000056
Accessibility   -0.001552
Name: MECOM, Length: 283, dtype: float32

In [63]:
hydra_celltype.gene_by_motif.data[['EVI1/MECOM']].sort_values(by='EVI1/MECOM', ascending=False).to_csv('./process_seacells/MECOM_socre.csv')

#### motif-gene visual

In [130]:
?hydra_celltype.plot_gene_motifs

Signature:
hydra_celltype.plot_gene_motifs(
    gene,
    motif=None,
    overwrite: bool = False,
    n: int = 10,
)
Docstring:
Plot top N motifs for a gene. Based on the absolute mean of the motif scores.

Parameters
----------
gene
    Gene name
motif
    Motif instance. Not used, self.motif is used instead.
overwrite
    Whether to overwrite existing plots, by default False
n
    Number of motifs to plot, by default 10
File:      ~/miniconda3/envs/get/lib/python3.12/site-packages/gcell/cell/celltype.py
Type:      method

In [211]:
hydra_celltype.motif.annotations.query('Motif.str.startswith("GATA")')

,Cluster_ID,Motif,Database,Consensus,Relative_orientation,Width,Left_offset,Right_offset,Name,DBD,Seed_motif,Total_width,Consensus_left,Consensus_right,Cluster_size
1948,242,GATA2_HUMAN.H11MO.0.A,HOCOMOCO_v11,CCTTATCTCCACCCCCCAG,-,19,3,0,GATA,C2H2,GATA1+TAL1_MA0140.2,22,4,14,23
1950,242,GATA1_HUMAN.H11MO.0.A,HOCOMOCO_v11,TCTTATCTGTCCCCGCCAG,-,19,3,0,GATA,C2H2,GATA1+TAL1_MA0140.2,22,4,14,23
1951,242,GATA1_MOUSE.H11MO.0.A,HOCOMOCO_v11,CCCTTATCTGCCCCCCCCA,-,19,2,1,GATA,C2H2,GATA1+TAL1_MA0140.2,22,4,14,23
1952,242,GATA4_HUMAN.H11MO.0.A,HOCOMOCO_v11,TCTTATCTGT,-,10,3,9,GATA,C2H2,GATA1+TAL1_MA0140.2,22,4,14,23
1953,242,GATA4_MOUSE.H11MO.0.A,HOCOMOCO_v11,TCTTATCTGT,-,10,3,9,GATA,C2H2,GATA1+TAL1_MA0140.2,22,4,14,23
1954,242,GATA6_MOUSE.H11MO.0.A,HOCOMOCO_v11,CCTTATCTCC,-,10,3,9,GATA,C2H2,GATA1+TAL1_MA0140.2,22,4,14,23
1955,242,GATA2_MOUSE.H11MO.0.A,HOCOMOCO_v11,TTCTTATCTGT,-,11,2,9,GATA,C2H2,GATA1+TAL1_MA0140.2,22,4,14,23
1956,242,GATA3_MOUSE.H11MO.0.A,HOCOMOCO_v11,GATTCTTATCTGT,-,13,0,9,GATA,C2H2,GATA1+TAL1_MA0140.2,22,4,14,23
1957,242,GATA3_HUMAN.H11MO.0.A,HOCOMOCO_v11,TTCTTATCTTT,-,11,2,9,GATA,C2H2,GATA1+TAL1_MA0140.2,22,4,14,23
1958,242,GATA6_HUMAN.H11MO.0.A,HOCOMOCO_v11,TTATCTTATCTGT,-,13,0,9,GATA,C2H2,GATA1+TAL1_MA0140.2,22,4,14,23


In [13]:
hydra_celltype.gene_by_motif.data.loc['BCL11A',:].sort_values(ascending=False)[:20]

HSF              5.325311e-07
SCRT1            4.617002e-07
Accessibility    4.586742e-07
NR/19            4.344989e-07
SMARCA1          4.232699e-07
GFI              3.351828e-07
ZIC              3.260163e-07
HD/18            3.206456e-07
GATA             3.093263e-07
ZNF708           2.972156e-07
LEF1             2.918574e-07
ETS/1            2.814297e-07
GLI              2.776497e-07
NR/11            2.709907e-07
HD/17            2.670934e-07
ZNF332           2.660777e-07
REST/NRSF        2.633404e-07
GRHL             2.608540e-07
HIC/2            2.592697e-07
HD/22            2.539518e-07
Name: HBG2, dtype: float32

In [ ]:
hydra_celltype.plot_gene_motifs('MECOM', n=11)

### peak-gene

In [46]:
len(hydra_celltype.peak_annot['Gene'].unique().tolist())

17233

In [47]:
hydra_celltype.peak_annot['Gene'].unique().tolist()

['A1BG-AS1',
 'A2M',
 'A2ML1-AS1',
 'A3GALT2',
 'A4GALT',
 'A4GNT',
 'AAAS',
 'AACS',
 'AADACL2-AS1',
 'AADACL4',
 'AADAT',
 'AAGAB',
 'AAK1',
 'AAMP',
 'AAR2',
 'AARD',
 'AARS2',
 'AARSD1',
 'AASDH',
 'AASS',
 'AATBC',
 'AATF',
 'AATK',
 'ABALON',
 'ABAT',
 'ABCA1',
 'ABCA13',
 'ABCA2',
 'ABCA3',
 'ABCA5',
 'ABCA6',
 'ABCA7',
 'ABCA9',
 'ABCB1',
 'ABCB10',
 'ABCB4',
 'ABCB6',
 'ABCB7',
 'ABCB8',
 'ABCB9',
 'ABCC1',
 'ABCC10',
 'ABCC11',
 'ABCC12',
 'ABCC3',
 'ABCC4',
 'ABCC5',
 'ABCC5-AS1',
 'ABCC6',
 'ABCC8',
 'ABCC9',
 'ABCD2',
 'ABCD3',
 'ABCD4',
 'ABCF1',
 'ABCF2',
 'ABCF3',
 'ABCG1',
 'ABCG2',
 'ABCG4',
 'ABCG5',
 'ABHD1',
 'ABHD10',
 'ABHD11',
 'ABHD12',
 'ABHD12B',
 'ABHD14B',
 'ABHD15',
 'ABHD15-AS1',
 'ABHD16A',
 'ABHD17A',
 'ABHD17B',
 'ABHD17C',
 'ABHD2',
 'ABHD3',
 'ABHD4',
 'ABHD5',
 'ABHD6',
 'ABHD8',
 'ABI1',
 'ABI2',
 'ABI3BP',
 'ABL1',
 'ABL2',
 'ABLIM1',
 'ABLIM2',
 'ABLIM3',
 'ABO',
 'ABR',
 'ABRA',
 'ABRACL',
 'ABRAXAS1',
 'ABRAXAS2',
 'ABT1',
 'ABTB1',
 'ABTB2',
 

In [39]:
data = pd.DataFrame(hydra_celltype.gene_jacobian_summary('GNAS', axis='region')).sort_values(by='Score', ascending=False)
data

,index,Chromosome,Start,End,Score
100,1576700,chr20,58839352,58839852,0.061522
102,1576702,chr20,58840468,58840968,0.004217
99,1576699,chr20,58838704,58839204,0.004132
149,1576749,chr20,58976232,58976728,0.003207
101,1576701,chr20,58839856,58840356,0.003147
...,...,...,...,...,...
705,1577305,chr20,58868632,58869132,0.000000
704,1577304,chr20,58864320,58864820,0.000000
703,1577303,chr20,58855804,58856304,0.000000
702,1577302,chr20,58854668,58855168,0.000000


In [ ]:
import pandas as pd

# Get a list of unique genes
gene_list = hydra_celltype.peak_annot['Gene'].unique().tolist()

# Initialize an empty list to store the results for each gene
df_list = []
i = 0

# Iterate through each gene
for gene in gene_list:
    i = i + 1
    print(i)
    # Extract Jacobian summary data and convert it to a DataFrame
    data = pd.DataFrame(hydra_celltype.gene_jacobian_summary(gene, axis='region'))
    
    # Sort the data by 'Score' in descending order
    data = data.sort_values(by='Score', ascending=False)
    
    # Keep only rows where Score > 0
    data = data[data['Score'] > 0]
    
    # Add a 'Gene' column
    data['Gene'] = gene
    
    # Append the result to the list
    df_list.append(data)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182


In [ ]:
final_df = pd.concat(df_list, axis=0, ignore_index=True)

In [ ]:
print(final_df)

In [ ]:
final_df.to_csv('peak_gene.csv')